# 03 · Table II: component ablation (cache only)

Rows 1-5 use the main draw (notebook 00); the pruning rows use the prefix draw
(notebook 01), whose unpruned majority is 73.92% and unpruned verifier 77.03%.
Accuracy and AUC are averaged over verifier-training seeds 0, 1, 2.

In [ ]:
#@title Setup: mount Drive, install lvpp, set cache paths
import os, sys, json, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "lvpp[gbt] @ git+https://github.com/KARTIKPANSURIYA/lvpp.git"], check=True)

# Root of the cached candidates/features. The default is the Drive layout the
# paper's runs wrote to; set LVPP_CACHE to use another location.
CACHE = os.environ.get("LVPP_CACHE", "/content/drive/MyDrive/lcldd_runs")
RESULTS = os.environ.get("LVPP_RESULTS", os.path.join(CACHE, "lvpp_results"))
os.makedirs(RESULTS, exist_ok=True)
PATHS = dict(
    main_pool   = f"{CACHE}/verifier_upgrade/pool.pt",       # 7,473 GSM8K train questions x 8
    main_test   = f"{CACHE}/verifier_upgrade/testfeat.pt",   # 1,319 GSM8K test questions x 8
    prefix_root = f"{CACHE}/abort_main_draw",                # prefix draw: pool/ and test/ shards
    span_root   = f"{CACHE}/span_fixed_vllm",                # Fig. 2 draw (1.5B-Instruct, corrected trigger)
    mi_pool     = f"{CACHE}/pending/mi_pool.pt",             # Math-Instruct on GSM8K
    mi_test     = f"{CACHE}/pending/mi_test.pt",
    math_pool   = f"{CACHE}/pending/math_pool.pt",           # Math-Instruct on MATH
    math_test   = f"{CACHE}/pending/math_test.pt",
    x15_pool    = f"{CACHE}/novelty/x15_pool.pt",            # 1.5B-Instruct, 1,200 questions
    x3_pool     = f"{CACHE}/novelty/x3_pool.pt",             # 3B-Instruct, 1,200 questions
    x3_test     = f"{CACHE}/novelty/x3_test.pt",             # 3B-Instruct, 400 test questions
    legacy_fewshot = f"{CACHE}/verifier_fewshot",            # Table IV, single-layer verifier
    legacy_mathmax = f"{CACHE}/verifier_mathmax",
    distilled_r32  = f"{CACHE}/verifier_final",
)
import numpy as np, pandas as pd, torch
from sklearn.metrics import roc_auc_score
print("device:", "cuda" if torch.cuda.is_available() else "cpu", "| cache:", CACHE)

In [ ]:
def report(rows, expected, keys, tol=0.6):
    """Print measured vs published values. Regenerated candidates are a new
    random draw, so agreement within about half a point is expected; from the
    released caches the numbers should match to the reported precision."""
    out = []
    for k in keys:
        m, e = rows.get(k), expected.get(k)
        if m is None or e is None:
            flag = ""
        elif e < 1e-3:                       # p-values: significant at the same level
            flag = "ok" if m < 1e-3 else "CHECK"
        elif e <= 1.0:                       # AUCs and rates
            flag = "ok" if abs(m - e) <= 0.01 else "CHECK"
        else:                                # accuracies in percentage points
            flag = "ok" if abs(m - e) <= tol else "CHECK"
        out.append(dict(quantity=k, measured=None if m is None else round(m, 4), paper=e, status=flag))
    df = pd.DataFrame(out)
    print(df.to_string(index=False))
    return df

In [ ]:
from lvpp.caches import load_pool_A, load_dict_B, load_prefix_C
from lvpp.selection import selection_hits
from lvpp.verifier import fit_verifier
from lvpp.pruning import prune_keep_and_cost, full_cost
from lvpp.stats import paired
SEEDS = [0, 1, 2]
te_full = load_dict_B(PATHS["main_test"], variants=("mid_mean", "upper_mean", "last_mean"))
D = te_full.P.shape[1] // 3
P_all, Z, y, _ = load_pool_A(PATHS["main_pool"], variants=("mid_mean", "upper_mean", "last_mean"))
h_maj = selection_hits(te_full, "majority")

In [ ]:
ARMS = {"mid layer only (L/2)": (slice(0, D), True),
        "last layer only (L)": (slice(2 * D, 3 * D), True),
        "multi-layer, without z(c)": (slice(0, 3 * D), False),
        "multi-layer with z(c)": (slice(0, 3 * D), True)}
rows = {"majority (no verifier)": 100 * h_maj.mean()}
for name, (sl, use_z) in ARMS.items():
    accs, aucs = [], []
    Ztr = Z if use_z else Z[:, :0]; Zte = te_full.Z if use_z else te_full.Z[:, :0]
    for s in SEEDS:
        v = fit_verifier(P_all[:, sl], Ztr, y, seed=s); sc = v.score(te_full.P[:, sl], Zte)
        accs.append(100 * selection_hits(te_full, "weighted", scores=sc).mean())
        aucs.append(roc_auc_score(te_full.y.numpy(), sc))
    rows[name] = float(np.mean(accs)); rows[name + " AUC"] = float(np.mean(aucs))
    print(f"{name:28s} {np.mean(accs):.2f} +/- {np.std(accs):.2f} | AUC {np.mean(aucs):.4f}")

In [ ]:
#@title Final generated token only (Section V-B text: +0.68 points, p = 0.79)
P_last, Zl, yl, _ = load_pool_A(PATHS["main_pool"], variants=("last_final",))
te_last = load_dict_B(PATHS["main_test"], variants=("last_final",))
v = fit_verifier(P_last, Zl, yl, seed=0)
h_last = selection_hits(te_last, "weighted", scores=v.score(te_last.P, te_last.Z))
r = paired(h_last, h_maj)
print(f"last token only: {r['acc']:.2f}% ({r['diff']:+.2f} vs majority, McNemar p = {r['p']:.2f})")

In [ ]:
#@title Pruning rows: prefix AUC and pruned selection (3 seeds)
PR = PATHS["prefix_root"]; K = 8; Bmax = 169
teF = load_prefix_C(PR, "test", Bmax); trF = load_prefix_C(PR, "pool", Bmax)
sF = fit_verifier(trF.P, trF.Z, trF.y, seed=0).score(teF.P, teF.Z)
h_maj_p = selection_hits(teF, "majority"); h_ver_p = selection_hits(teF, "weighted", scores=sF)
c_full = full_cost(teF.extra["n_total"])
print(f"prefix draw: majority {100*h_maj_p.mean():.2f}% | unpruned verifier {100*h_ver_p.mean():.2f}% | {c_full:.0f} tokens/question")
cache = {}
for B, rho in [(169, 0.25), (110, 0.25), (110, 0.50), (126, 0.50)]:
    if B not in cache: cache[B] = (load_prefix_C(PR, "test", B), load_prefix_C(PR, "pool", B))
    teB, trB = cache[B]; accs, aucs = [], []
    for s in SEEDS:
        sB = fit_verifier(trB.P, trB.Z, trB.y, seed=s).score(teB.P, teB.Z)
        aucs.append(roc_auc_score(teB.y.numpy(), sB))
        keep, cost = prune_keep_and_cost(sB, teB.extra["finished"], teF.extra["n_total"], B, rho)
        accs.append(100 * selection_hits(teF, "weighted", scores=sF, keep=keep).mean())
    rows[f"prune B={B} rho={rho}"] = float(np.mean(accs))
    rows[f"prune B={B} AUC"] = float(np.mean(aucs))
    rows[f"prune B={B} rho={rho} saved"] = 100 * (1 - cost / c_full)
    print(f"B={B} rho={rho:.2f}: {np.mean(accs):.2f} +/- {np.std(accs):.2f} | prefix AUC {np.mean(aucs):.4f} | saved {100*(1-cost/c_full):.1f}%")

In [ ]:
EXPECTED = {"majority (no verifier)": 74.91,
            "mid layer only (L/2)": 76.75, "mid layer only (L/2) AUC": 0.842,
            "last layer only (L)": 77.79, "last layer only (L) AUC": 0.844,
            "multi-layer, without z(c)": 78.90, "multi-layer, without z(c) AUC": 0.868,
            "multi-layer with z(c)": 78.92, "multi-layer with z(c) AUC": 0.870,
            "prune B=169 rho=0.25": 77.13, "prune B=169 AUC": 0.843, "prune B=169 rho=0.25 saved": 3.8,
            "prune B=110 rho=0.25": 76.72, "prune B=110 AUC": 0.827, "prune B=110 rho=0.25 saved": 8.0,
            "prune B=110 rho=0.5": 76.55, "prune B=110 rho=0.5 saved": 16.3}
report(rows, EXPECTED, list(EXPECTED)).to_csv(f"{RESULTS}/table2.csv", index=False)